In [1]:
import os
import numpy as np
import torch
import torch.nn as nn

from utils import evaluate
from torchvision import datasets
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

weights_path = "models_weights"
os.makedirs(weights_path, exist_ok=True)  # torch.save non crea la cartella se manca

# Dataset

In [3]:
train_ds = datasets.CIFAR10(root='datasets', train=True, download=True)
test_ds = datasets.CIFAR10(root='datasets', train=False, download=True)
class_size = 500

(x_train, y_train), (x_test, y_test) = (train_ds.data, np.array(train_ds.targets)), (test_ds.data, np.array(test_ds.targets))
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

# TASK TARGET: CAT (3) VS DOG (5), CON SOLO class_size IMMAGINI PER CLASSE
# TASK SORGENTE: LE ALTRE 8 CLASSI, CON TUTTE LE IMMAGINI
tgt_classes = [3, 5]
src_classes = [c for c in range(10) if c not in tgt_classes]
print("TARGET:", [class_names[c] for c in tgt_classes])
print("SOURCE:", [class_names[c] for c in src_classes])

I_train0 = np.where(y_train==tgt_classes[0])
I_train1 = np.where(y_train==tgt_classes[1])
I_train = np.concatenate((I_train0[0][0:class_size],I_train1[0][0:class_size]))
y_train_tgt = [0]*class_size+[1]*class_size
y_train_tgt = np.array(y_train_tgt)
x_train_tgt = x_train[I_train]
I_test_tgt = np.isin(y_test, tgt_classes)
y_test_tgt = (y_test[I_test_tgt] == tgt_classes[1]).astype(np.int64)  # cat -> 0, dog -> 1
x_test_tgt = x_test[I_test_tgt]

x_train_tgt = x_train_tgt.astype('float32')
x_test_tgt = x_test_tgt.astype('float32')
x_train_tgt /= 255      # 32x32x3
x_test_tgt /= 255
x_train_tgt = x_train_tgt.transpose(0, 3, 1, 2)  # channels-first: (N, 32, 32, 3) -> (N, 3, 32, 32)
x_test_tgt = x_test_tgt.transpose(0, 3, 1, 2)

I_train_src = ~np.isin(y_train, tgt_classes)
y_train_src = np.searchsorted(src_classes, y_train[I_train_src])  # etichette rimappate in 0..7
x_train_src = x_train[I_train_src]

I_test_src = ~np.isin(y_test, tgt_classes)
y_test_src = np.searchsorted(src_classes, y_test[I_test_src])
x_test_src = x_test[I_test_src]

x_train_src = x_train_src.astype('float32')
x_test_src = x_test_src.astype('float32')
x_train_src /= 255
x_test_src /= 255
x_train_src = x_train_src.transpose(0, 3, 1, 2)
x_test_src = x_test_src.transpose(0, 3, 1, 2)

# VALIDATION SET: 20% DEL TRAINING, ESTRATTO A CASO (IL TEST SET SI USA SOLO PER LA VALUTAZIONE FINALE)
rng = np.random.default_rng(0)
def train_val_split(x, y, val_fraction=0.2):
    idx = rng.permutation(len(x))
    n_val = int(len(x) * val_fraction)
    return x[idx[n_val:]], y[idx[n_val:]], x[idx[:n_val]], y[idx[:n_val]]

x_train_tgt, y_train_tgt, x_val_tgt, y_val_tgt = train_val_split(x_train_tgt, y_train_tgt)
x_train_src, y_train_src, x_val_src, y_val_src = train_val_split(x_train_src, y_train_src)

print("SHAPE TRAIN TARGET: ",x_train_tgt.shape)
print("SHAPE VAL TARGET: ",x_val_tgt.shape)
print("SHAPE TEST TARGET: ",x_test_tgt.shape)
print("SHAPE TRAIN SOURCE: ",x_train_src.shape)
print("SHAPE VAL SOURCE: ",x_val_src.shape)
print("SHAPE TEST SOURCE: ",x_test_src.shape)

TARGET: ['cat', 'dog']
SOURCE: ['airplane', 'automobile', 'bird', 'deer', 'frog', 'horse', 'ship', 'truck']
SHAPE TRAIN TARGET:  (800, 3, 32, 32)
SHAPE VAL TARGET:  (200, 3, 32, 32)
SHAPE TEST TARGET:  (2000, 3, 32, 32)
SHAPE TRAIN SOURCE:  (32000, 3, 32, 32)
SHAPE VAL SOURCE:  (8000, 3, 32, 32)
SHAPE TEST SOURCE:  (8000, 3, 32, 32)


### TRAINING LENET ON THE 8 SOURCE CLASSES

![title](LeNet-5_architecture.png)

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')

EPOCHS = 10
LR = 5e-4
TRAIN = True

In [5]:
def fit(model, optimizer, loss_fn, x, y, epochs, validation_data, scheduler=None, verbose=1, batch_size=32):

    x = torch.as_tensor(x, dtype=torch.float32)
    y = torch.as_tensor(y, dtype=torch.long)
    history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(x))
        loss_sum, correct = 0.0, 0
        for i in range(0, len(x), batch_size):
            idx = perm[i:i + batch_size]
            xb, yb = x[idx].to(device), y[idx].to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(xb)
            correct += (logits.argmax(1) == yb).sum().item()
        val_loss, val_acc = evaluate(model, *validation_data, loss_fn, batch_size)
        for k, v in zip(history, (loss_sum / len(x), correct / len(x), val_loss, val_acc)):
            history[k].append(v)
        if verbose:
            print(f"Epoch {epoch + 1}/{epochs} - loss: {history['loss'][-1]:.4f} - accuracy: {history['accuracy'][-1]:.4f}"
                  f" - val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}")
        if scheduler is not None:
            old_lr = optimizer.param_groups[0]['lr']
            scheduler.step(val_acc)
            new_lr = optimizer.param_groups[0]['lr']
            if verbose and new_lr < old_lr:
                print(f'ReduceLROnPlateau: learning rate ridotto a {new_lr:.1e}')
    return history

def evaluate(model, x, y, loss_fn, batch_size=32):
    x = torch.as_tensor(x, dtype=torch.float32)
    y = torch.as_tensor(y, dtype=torch.long)
    model.eval()
    loss_sum, correct = 0.0, 0
    with torch.no_grad():
        for i in range(0, len(x), batch_size):
            xb, yb = x[i:i + batch_size].to(device), y[i:i + batch_size].to(device)
            logits = model(xb)
            loss_sum += loss_fn(logits, yb).item() * len(xb)
            correct += (logits.argmax(1) == yb).sum().item()
    return loss_sum / len(x), correct / len(x)

In [6]:
# RETE ADDESTRATA SULLE 8 CLASSI SORGENTE
lenet = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=5, padding='same'), nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(32, 48, kernel_size=5, padding='valid'), nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(48 * 6 * 6, 256), nn.ReLU(),  # 32 -> pool 16 -> conv 12 -> pool 6
    nn.Linear(256, 84), nn.ReLU(),
    nn.Linear(84, 8)
).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = Adam(lenet.parameters(), lr=LR)
reduce_lr = ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.2, min_lr=1e-6)  # monitora la val_accuracy

name = "lenet_cifar_src"

exists = os.path.isfile(f'{weights_path}/{name}.pt')
if exists and not TRAIN:
    lenet.load_state_dict(torch.load(f'{weights_path}/{name}.pt', map_location=device))
else:
    fit(lenet, optimizer, loss_fn, x_train_src, y_train_src, epochs=EPOCHS, validation_data=(x_val_src, y_val_src), scheduler=reduce_lr)
    torch.save(lenet.state_dict(), f'{weights_path}/{name}.pt')

src_loss, src_acc = evaluate(lenet, x_test_src, y_test_src, loss_fn)
print(f"TEST ACCURACY SOURCE (8 classi): {src_acc:.4f}")

Epoch 1/10 - loss: 1.4475 - accuracy: 0.4421 - val_loss: 1.1927 - val_accuracy: 0.5544
Epoch 2/10 - loss: 1.0884 - accuracy: 0.5986 - val_loss: 0.9804 - val_accuracy: 0.6492
Epoch 3/10 - loss: 0.9115 - accuracy: 0.6687 - val_loss: 0.8443 - val_accuracy: 0.7004
Epoch 4/10 - loss: 0.8070 - accuracy: 0.7093 - val_loss: 0.7827 - val_accuracy: 0.7183
Epoch 5/10 - loss: 0.7165 - accuracy: 0.7415 - val_loss: 0.7817 - val_accuracy: 0.7206
Epoch 6/10 - loss: 0.6492 - accuracy: 0.7682 - val_loss: 0.7578 - val_accuracy: 0.7296
Epoch 7/10 - loss: 0.5882 - accuracy: 0.7897 - val_loss: 0.7879 - val_accuracy: 0.7251
Epoch 8/10 - loss: 0.5272 - accuracy: 0.8108 - val_loss: 0.7378 - val_accuracy: 0.7428
Epoch 9/10 - loss: 0.4635 - accuracy: 0.8344 - val_loss: 0.7351 - val_accuracy: 0.7501
Epoch 10/10 - loss: 0.4120 - accuracy: 0.8535 - val_loss: 0.7260 - val_accuracy: 0.7542
TEST ACCURACY SOURCE (8 classi): 0.7536


### TRAINING LENET ON CAT & DOG - ONLY HEAD

In [7]:
# RETE CHE UTILIZZA I PESI OTTENUTI DALLA RETE PRECEDENTE AI LIVELLI CONVOLUZIONALI
lenet_head = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=5, padding='same'), nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(32, 48, kernel_size=5, padding='valid'), nn.ReLU(),
    nn.MaxPool2d(2),
)
lenet_head.load_state_dict(lenet[0:6].state_dict())

# NEI LIVELLI DENSI PESI DIVERSI
lenet_head.append(nn.Flatten())
lenet_head.append(nn.Linear(48 * 6 * 6, 256)); lenet_head.append(nn.ReLU())
lenet_head.append(nn.Linear(256, 84)); lenet_head.append(nn.ReLU())
lenet_head.append(nn.Linear(84, 2))
lenet_head = lenet_head.to(device)

# IN FASE DI ADDESTRAMENTO I PESI DELLA PARTE CONVOLUZIONALE NON SONO MODIFICABILI
for param in lenet_head[0:6].parameters():
    param.requires_grad = False

optimizer_head = Adam([p for p in lenet_head.parameters() if p.requires_grad], lr=LR)
reduce_lr_head = ReduceLROnPlateau(optimizer_head, mode='max', patience=3, factor=0.2, min_lr=1e-6)


name = "lenet_cifar_only_head"

exists = os.path.isfile(f'{weights_path}/{name}.pt')
if exists and not TRAIN:
    lenet_head.load_state_dict(torch.load(f'{weights_path}/{name}.pt', map_location=device))
else:
    fit(lenet_head, optimizer_head, loss_fn, x_train_tgt, y_train_tgt, epochs=EPOCHS//2, validation_data=(x_val_tgt, y_val_tgt), scheduler=reduce_lr_head)
    torch.save(lenet_head.state_dict(), f'{weights_path}/{name}.pt')

Epoch 1/5 - loss: 0.6644 - accuracy: 0.5988 - val_loss: 0.6364 - val_accuracy: 0.6750
Epoch 2/5 - loss: 0.5704 - accuracy: 0.7288 - val_loss: 0.6101 - val_accuracy: 0.6750
Epoch 3/5 - loss: 0.4989 - accuracy: 0.7612 - val_loss: 0.6162 - val_accuracy: 0.6550
Epoch 4/5 - loss: 0.3972 - accuracy: 0.8325 - val_loss: 0.6351 - val_accuracy: 0.6650
Epoch 5/5 - loss: 0.3243 - accuracy: 0.8688 - val_loss: 0.7849 - val_accuracy: 0.6400
ReduceLROnPlateau: learning rate ridotto a 1.0e-04


### TRAINING LENET ON CAT & DOG - FINE-TUNING


In [8]:
# RETE UGUALE ALLA PRECEDENTE PER ARCHITETTURA MA CON PESI LIBERI
lenet_ft = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=5, padding='same'), nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(32, 48, kernel_size=5, padding='valid'), nn.ReLU(),
    nn.MaxPool2d(2),
)
lenet_ft.load_state_dict(lenet[0:6].state_dict())

lenet_ft.append(nn.Flatten())
lenet_ft.append(nn.Linear(48 * 6 * 6, 256)); lenet_ft.append(nn.ReLU())
lenet_ft.append(nn.Linear(256, 84)); lenet_ft.append(nn.ReLU())
lenet_ft.append(nn.Linear(84, 2))
lenet_ft = lenet_ft.to(device)

optimizer_ft = Adam(lenet_ft.parameters(), lr=LR)
reduce_lr_ft = ReduceLROnPlateau(optimizer_ft, mode='max', patience=3, factor=0.2, min_lr=1e-6)

# LE ULTIME DUE RETI SONO ADDESTRATE PER LO STESSO NUMERO DI EPOCHE
name = "lenet_cifar_fine_tuning"

exists = os.path.isfile(f'{weights_path}/{name}.pt')
if exists and not TRAIN:
    lenet_ft.load_state_dict(torch.load(f'{weights_path}/{name}.pt', map_location=device))
else:
    fit(lenet_ft, optimizer_ft, loss_fn, x_train_tgt, y_train_tgt, epochs=EPOCHS//2, validation_data=(x_val_tgt, y_val_tgt), scheduler=reduce_lr_ft)
    torch.save(lenet_ft.state_dict(), f'{weights_path}/{name}.pt')

Epoch 1/5 - loss: 0.6716 - accuracy: 0.5575 - val_loss: 0.6359 - val_accuracy: 0.6800
Epoch 2/5 - loss: 0.5662 - accuracy: 0.7063 - val_loss: 0.6187 - val_accuracy: 0.6650
Epoch 3/5 - loss: 0.4643 - accuracy: 0.7963 - val_loss: 0.6309 - val_accuracy: 0.6600
Epoch 4/5 - loss: 0.3882 - accuracy: 0.8300 - val_loss: 0.6632 - val_accuracy: 0.6800
Epoch 5/5 - loss: 0.3079 - accuracy: 0.8788 - val_loss: 0.7322 - val_accuracy: 0.6800
ReduceLROnPlateau: learning rate ridotto a 1.0e-04


In [9]:
b = evaluate(lenet_head, x_test_tgt, y_test_tgt, loss_fn)
c = evaluate(lenet_ft, x_test_tgt, y_test_tgt, loss_fn)

print(f"{'':<24}{'LOSS':>8}{'ACCURACY':>10}")
print(f"{'lenet_cifar_only_head':<24}{b[0]:>8.4f}{b[1]:>10.4f}")
print(f"{'lenet_cifar_fine_tuning':<24}{c[0]:>8.4f}{c[1]:>10.4f}")

                            LOSS  ACCURACY
lenet_cifar_only_head     0.7442    0.6505
lenet_cifar_fine_tuning   0.7040    0.6770


In [10]:
# BASELINE: STESSA ARCHITETTURA ADDESTRATA DA ZERO SOLO SU CAT & DOG (NESSUN TRANSFER)
lenet_scratch = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=5, padding='same'), nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(32, 48, kernel_size=5, padding='valid'), nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(48 * 6 * 6, 256), nn.ReLU(),
    nn.Linear(256, 84), nn.ReLU(),
    nn.Linear(84, 2)
).to(device)

optimizer_scratch = Adam(lenet_scratch.parameters(), lr=LR)
reduce_lr_scratch = ReduceLROnPlateau(optimizer_scratch, mode='max', patience=3, factor=0.2, min_lr=1e-6)

# STESSI DATI E STESSO NUMERO DI EPOCHE DELLE DUE RETI CON TRANSFER
name = "lenet_cifar_scratch"

exists = os.path.isfile(f'{weights_path}/{name}.pt')
if exists and not TRAIN:
    lenet_scratch.load_state_dict(torch.load(f'{weights_path}/{name}.pt', map_location=device))
else:
    fit(lenet_scratch, optimizer_scratch, loss_fn, x_train_tgt, y_train_tgt, epochs=EPOCHS//2, validation_data=(x_val_tgt, y_val_tgt), scheduler=reduce_lr_scratch)
    torch.save(lenet_scratch.state_dict(), f'{weights_path}/{name}.pt')

a = evaluate(lenet_scratch, x_test_tgt, y_test_tgt, loss_fn)

print()
print(f"{'':<24}{'LOSS':>8}{'ACCURACY':>10}")
print(f"{'lenet_cifar_scratch':<24}{a[0]:>8.4f}{a[1]:>10.4f}")
print(f"{'lenet_cifar_only_head':<24}{b[0]:>8.4f}{b[1]:>10.4f}")
print(f"{'lenet_cifar_fine_tuning':<24}{c[0]:>8.4f}{c[1]:>10.4f}")

Epoch 1/5 - loss: 0.6941 - accuracy: 0.4850 - val_loss: 0.6914 - val_accuracy: 0.5150
Epoch 2/5 - loss: 0.6943 - accuracy: 0.5275 - val_loss: 0.6883 - val_accuracy: 0.4700
Epoch 3/5 - loss: 0.6924 - accuracy: 0.4938 - val_loss: 0.6899 - val_accuracy: 0.5200
Epoch 4/5 - loss: 0.6810 - accuracy: 0.5975 - val_loss: 0.7163 - val_accuracy: 0.5250
Epoch 5/5 - loss: 0.6735 - accuracy: 0.5637 - val_loss: 0.6822 - val_accuracy: 0.5750

                            LOSS  ACCURACY
lenet_cifar_scratch       0.6837    0.5605
lenet_cifar_only_head     0.7442    0.6505
lenet_cifar_fine_tuning   0.7040    0.6770
